In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

torch.manual_seed(1)

In [ ]:
# LSTM : 시간 순서가 있는 데이터를 처리하는 레이어
# 일반 신경망은 이전 입력을 기억 못하는데, LSTM은 기억

# 레이어 설계
# 특징수 예시 : 전류, RPM
lstm = nn.LSTM(3, 3)
#              ↑  ↑
#          특징수  기억크기
inputs = [torch.randn(1, 1, 3) for _ in range(5)]

hidden = (torch.rand(1, 1, 3), torch.rand(1, 1, 3))

for i in inputs:
  # 레이어 실행
  out, hidden = lstm(i.view(1, 1, -1), hidden)
  #                  ↑                 ↑
  #                3D 입력형식          이전 기억
  # 3D로 만드는 이유는 lstm이 받을 수 있는 형태이기 때문이다.

# 위의 과정과 같지만 코드 줄을 줄이는 방법 (for문을 사용하지 않음)
inputs = torch.cat(inputs).view(len(inputs), 1, -1)
hidden = (torch.rand(1, 1, 3), torch.rand(1, 1, 3))

out, hidden = lstm(inputs, hidden)

In [ ]:
# 단어를 숫자로 바꾸는 함수
# sequence : 순서 있는 데이터(LSTM의 핵심)
def prepare_sequence(seq, to_ix):
  idxs = [to_ix[w] for w in seq]
  return torch.tensor(idxs, dtype=torch.long)

training_data = [
  # 첫 번째 : 순서 있는 입력 데이터
  # 두 번째 : 각 입력에 대한 정답 레이블
  ("The dog ate the apple.".split(), ["DET", "nn", "V", "DET", "NN"]),
  ("Everybody read that book".split(), ["NN", "V", "DET", "NN"])
]

# 단어를 index화
word_to_ix = {}

# 중복되는 단어를 제외하고 단어의 순서를 index화
for sent, tags in training_data:
  for word in sent:
    if word not in word_to_ix:
      word_to_ix[word] = len(word_to_ix)


print(word_to_ix)

# 품사를 숫자로 바꾸는 딕셔너리
tag_to_ix = {"DET": 0, "NN": 1, "V": 2}

# 단어를 몇 차원 벡터로 표현할지 크기
EMBEDDING_DIM = 6
# 기억 크기
HIDDEN_DIM = 6

In [ ]:
class LSTMTagger(nn.Module):
  def __init__(self, embedding_dim, hidden_dim, vocab_size, target_size):
    super().__init__()
    self.hidden_dim = hidden_dim

    # Embedding 단어를 벡터로 바꾸는 것
    self.word_embeddings = nn.Embedding(vocab_size, embedding_dim)

    # embedding_dim
    self.lstm = nn.LSTM(embedding_dim, hidden_dim)

    # 기억 크기를 품사의 정답에 맞게 압축
    self.hidden2tag = nn.Linear(hidden_dim, target_size)